In [75]:
import pm4py
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.objects.log.exporter.xes import exporter as xes_exporter
from pm4py.objects.log.obj import EventLog
from copy import deepcopy


In [76]:
thesis_working_xes = "D:\\LTNcoder\\.out\eventlogs\\thesis_working.xes"

<>:1: DeprecationWarning: invalid escape sequence \e
<>:1: DeprecationWarning: invalid escape sequence \e
C:\Users\devas\AppData\Local\Temp\ipykernel_50356\3760449357.py:1: DeprecationWarning: invalid escape sequence \e
  thesis_working_xes = "D:\\LTNcoder\\.out\eventlogs\\thesis_working.xes"


In [77]:
read_log = xes_importer.apply(thesis_working_xes)
print(f"Log imported with {len(read_log)} traces.")

parsing log, completed traces ::   0%|          | 0/3 [00:00<?, ?it/s]

'Log imported with 3 traces.'


In [78]:
# print(read_log[0])
# type(read_log[0].attributes["concept:name"])
# print(len(read_log[0]))
# for i in range(len(read_log[0])):
#     print(read_log[0][i]["case_id"])
# read_log[0][0]["case_id"] = 0
# for event in read_log[0]:
#     print(event["case_id"])
#     print(event["@@index"])
#     print(event["@@case_index"])

    

In [82]:
new_log = EventLog()
event_id = 0
case_id = 0
for case_id in range(100):
    new_trace = deepcopy(read_log[0])
    new_trace.attributes["concept:name"] = str(case_id + 1)
    for j in range(len(new_trace)):
        new_trace[j]["case_id"] = case_id + 1
        new_trace[j]["@@index"] = event_id
        new_trace[j]["@@case_index"] = case_id
        event_id += 1
    new_log.append(new_trace)
for case_id in range(100, 130):
    new_trace = deepcopy(read_log[1])
    new_trace.attributes["concept:name"] = str(case_id + 1)
    for j in range(len(new_trace)):
        new_trace[j]["case_id"] = case_id + 1
        new_trace[j]["@@index"] = event_id
        new_trace[j]["@@case_index"] = case_id
        event_id += 1
        
    new_log.append(new_trace)
for case_id in range(130, 140):
    new_trace = deepcopy(read_log[2])
    new_trace.attributes["concept:name"] = str(case_id + 1)
    for j in range(len(new_trace)):
        new_trace[j]["case_id"] = case_id + 1
        new_trace[j]["@@index"] = event_id
        new_trace[j]["@@case_index"] = case_id
        event_id += 1
        
    new_log.append(new_trace)

  

In [83]:
print(new_log)

[{'attributes': {'concept:name': '1'}, 'events': [{'anomaly': 'normal', 'case_id': 1, 'org:resource': 'Sherlene', 'concept:name': 'Identify Problem', 'time:timestamp': datetime.datetime(2025, 4, 22, 16, 40, 24, 616625, tzinfo=datetime.timezone.utc), '@@index': 0, '@@case_index': 0}, '..', {'anomaly': 'normal', 'case_id': 1, 'org:resource': 'Lucy', 'concept:name': 'Final Decision', 'time:timestamp': datetime.datetime(2025, 4, 22, 18, 0, 24, 616625, tzinfo=datetime.timezone.utc), '@@index': 8, '@@case_index': 0}]}, '....', {'attributes': {'concept:name': '140'}, 'events': [{'anomaly': 'normal', 'case_id': 140, 'org:resource': 'Sherlene', 'concept:name': 'Identify Problem', 'time:timestamp': datetime.datetime(2025, 4, 22, 16, 40, 24, 616625, tzinfo=datetime.timezone.utc), '@@index': 1191, '@@case_index': 139}, '..', {'anomaly': 'normal', 'case_id': 140, 'org:resource': 'Lucy', 'concept:name': 'Final Decision', 'time:timestamp': datetime.datetime(2025, 4, 22, 18, 0, 24, 616625, tzinfo=date

In [87]:
print(len(new_log))
print(len(read_log))


140
3


In [88]:
xes_exporter.apply(new_log, "D:\\LTNcoder\\.out\\eventlogs\\thesis_working_new.xes")

exporting log, completed traces ::   0%|          | 0/140 [00:00<?, ?it/s]

In [89]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser

In [90]:
event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log("D:\\LTNcoder\\.out\\eventlogs\\thesis_working_new.xes")

parsing log, completed traces ::   0%|          | 0/140 [00:00<?, ?it/s]

In [91]:


discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=2)
declare_model: DeclareModel = discovery.run()
print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
model_constraints = declare_model.get_decl_model_constraints()

basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
conf_check_res: MPDeclareResultsBrowser = basic_checker.run()

Computing discovery ...
'Total constraints discovered: 953'


In [92]:
import pandas as pd
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)
import pandas as pd
import numpy as np

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    'activation_rate': activation_rate
})

C:\Users\devas\AppData\Local\Temp\ipykernel_50356\1044401953.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)


In [93]:
metrics_df

,support,confidence,activation_rate
Existence1[Submit] | |,1.000000,0.000000,0.000000
Absence2[Submit] | |,1.000000,0.000000,0.000000
Exactly1[Submit] | |,1.000000,0.000000,0.000000
Existence1[Review] | |,1.000000,0.000000,0.000000
Absence2[Review] | |,1.000000,0.000000,0.000000
...,...,...,...
"Not Precedence[Develop Method, Submit] | |",0.928571,0.928571,1.000000
"Not Chain Response[Submit, Develop Method] | |",1.000000,1.000000,1.000000
"Not Chain Response[Develop Method, Submit] | |",0.071429,1.000000,0.071429
"Not Chain Precedence[Submit, Develop Method] | |",0.071429,1.000000,0.071429


In [94]:
state_df = conf_check_res.get_metric(metric="state")
state_df

,Existence1[Submit] | |,Absence2[Submit] | |,Exactly1[Submit] | |,Existence1[Review] | |,Absence2[Review] | |,Exactly1[Review] | |,Existence1[Final Decision] | |,Absence2[Final Decision] | |,Exactly1[Final Decision] | |,End[Final Decision] | |,...,"Precedence[Develop Method, Submit] | |","Alternate Precedence[Develop Method, Submit] | |","Not Responded Existence[Submit, Develop Method] | |","Not Response[Submit, Develop Method] | |","Not Precedence[Submit, Develop Method] | |","Not Precedence[Develop Method, Submit] | |","Not Chain Response[Submit, Develop Method] | |","Not Chain Response[Develop Method, Submit] | |","Not Chain Precedence[Submit, Develop Method] | |","Not Chain Precedence[Develop Method, Submit] | |"
0,1,1,1,1,1,1,1,1,1,1,...,0,0,1,1,0,1,1,0,0,1
1,1,1,1,1,1,1,1,1,1,1,...,0,0,1,1,0,1,1,0,0,1
2,1,1,1,1,1,1,1,1,1,1,...,0,0,1,1,0,1,1,0,0,1
3,1,1,1,1,1,1,1,1,1,1,...,0,0,1,1,0,1,1,0,0,1
4,1,1,1,1,1,1,1,1,1,1,...,0,0,1,1,0,1,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,1,1,1,1,1,1,1,1,1,1,...,1,1,0,1,1,0,1,1,1,1
136,1,1,1,1,1,1,1,1,1,1,...,1,1,0,1,1,0,1,1,1,1
137,1,1,1,1,1,1,1,1,1,1,...,1,1,0,1,1,0,1,1,1,1
138,1,1,1,1,1,1,1,1,1,1,...,1,1,0,1,1,0,1,1,1,1
